# Instruction
1. Get the openshift token and login
2. Execute `oc port-forward deployment/open-webui-mastering-homework 8000:8000 -n rit-genai-naga-dev`


## 1. Health check

In [5]:
import requests
import time
BASE = "http://localhost:8000"

r = requests.get(f"{BASE}/")
print(r.status_code, r.json())

200 {'status': 'ok'}


## 2. Upload question PDF → Markdown

Set `PDF_PATH`, `GROUP_ID`, and `MODEL_ID` before running.

In [11]:
PDF_PATH = '/Users/thy/Downloads/1a75e71e-1257-4852-a3ad-d22059ee3616_Fall_25_Calculus_1 Homework 9 Section 21.pdf'   # <-- change this
GROUP_ID = "1cf1b19f-f173-4f64-a050-338a54924325"          # <-- change this
MODEL_ID = "-math1011-spring-2026-hw-9-model"          # <-- change this

with open(PDF_PATH, "rb") as f:
    r = requests.post(
        f"{BASE}/homework/pdf-to-markdown",
        params={"doc_type": "question", "group_id": GROUP_ID, "model_id": MODEL_ID},
        files={"file": ("homework.pdf", f, "application/pdf")},
    )

print(r.status_code, r.json())
job_id = r.json()["job_id"]
print("job_id:", job_id)

200 {'job_id': '176a1f97-c681-4dbc-bfb6-94db1489714a', 'status': 'queued', 'message': 'PDF conversion started. Poll GET /pipeline/status/{job_id} for progress.'}
job_id: 176a1f97-c681-4dbc-bfb6-94db1489714a


## 3. Poll job status until done or failed

In [12]:
# job_id = "paste-job-id-here"  # or use the one from cell above

while True:
    r = requests.get(f"{BASE}/pipeline/status/{job_id}")
    data = r.json()
    print(data["status"], "—", data.get("error") or "")
    if data["status"] in ("done", "failed"):
        break
    time.sleep(10)

print("\nFinal:", data)

running — 
running — 
running — 
running — 
running — 
running — 
running — 
done — 

Final: {'job_id': '176a1f97-c681-4dbc-bfb6-94db1489714a', 'step': 'pdf_to_markdown', 'homework_id': 'fc23d5e4-4f6f-4f32-b95c-eee747063335', 'status': 'done', 'error': None, 'created_at': '2026-03-30T05:36:53.830415', 'finished_at': '2026-03-30T05:38:04.384174'}


## 4. Check homework (question + topic mapping)

In [13]:
# Use homework_id from the completed job
homework_id = data["homework_id"]

r = requests.get(f"{BASE}/homework/", params={"homework_id": homework_id})
hw = r.json()[0]
print("question_uploaded:", hw["question_uploaded"])
print("topic_mapped:", hw["topic_mapped"])
print("\n--- Question Data (first 500 chars) ---")
print(hw["question_data"][:500])
print("\n--- Topic Mapping ---")
import json
print(json.dumps(hw["topic_mapping"], indent=2))

question_uploaded: True
topic_mapped: True

--- Question Data (first 500 chars) ---
```markdown
# MATH-UA 121: Calculus I  
**Fall 25**  

---

## Written Assignment 9  
**Due: Sunday, November 7th via Gradescope**

---

### Instructions

- This homework should be submitted via Gradescope by the date listed above.
- There are three main ways you might want to write up your work:
  - Write on this pdf using a tablet
  - Print this worksheet and write in the space provided
  - Write your answers on paper, clearly numbering each question and part.
    - *If using either of the las

--- Topic Mapping ---
{
  "1": [
    "Limits at Infinity"
  ],
  "2": [
    "Limits"
  ],
  "3": [
    "Critical Numbers",
    "Absolute Value Functions"
  ],
  "4": [
    "Critical Numbers",
    "Product Rule"
  ],
  "5": [
    "Critical Numbers",
    "Logarithmic Functions"
  ],
  "6": [
    "Critical Numbers",
    "Derivative of Inverse Trigonometric Functions"
  ],
  "7": [
    "Critical Numbers",
    "Quot

## 5. Export student conversations

In [18]:
r = requests.post(f"{BASE}/conversation/export", params={"homework_id": homework_id})
print(r.status_code, r.json())

200 {'status': 'success', 'homework_id': 'fc23d5e4-4f6f-4f32-b95c-eee747063335', 'group_id': '1cf1b19f-f173-4f64-a050-338a54924325', 'model_id': '-math1011-spring-2026-hw-9-model', 'total_students': 2, 'students': [{'student_id': 'cd6cfa15-7960-47b6-a802-7daf8bea8609', 'student_email': 'mathstudent1@gmail.com', 'conversation_count': 5, 'message_count': 44}, {'student_id': 'c9efa19c-c395-468d-9e3e-45b04865243c', 'student_email': 'mathstudent2@gmail.com', 'conversation_count': 3, 'message_count': 38}]}


## 6. Run analysis

In [19]:
homework_id

'fc23d5e4-4f6f-4f32-b95c-eee747063335'

In [20]:
r = requests.post(f"{BASE}/analysis/run", params={"homework_id": homework_id})
print(r.status_code, r.json())
analysis_job_id = r.json()["job_id"]

200 {'job_id': 'd724b515-85ac-43f7-ad4e-1a5f67bc50e5', 'status': 'queued', 'message': 'Analysis started. Poll GET /pipeline/status/{job_id} for progress.'}


In [21]:
# Poll analysis job
while True:
    r = requests.get(f"{BASE}/pipeline/status/{analysis_job_id}")
    data = r.json()
    print(data["status"], "—", data.get("error") or "")
    if data["status"] in ("done", "failed"):
        break
    time.sleep(10)

running — 
running — 
done — 


## 7. View analysis results

In [22]:
r = requests.get(f"{BASE}/analysis/", params={"homework_id": homework_id})
results = r.json()
print(f"{len(results)} students analyzed")
for s in results:
    print(f"  {s['student_email']}: {s['total_solved']}/{s['total_question']} solved")

2 students analyzed
  mathstudent2@gmail.com: 6/19 solved
  mathstudent1@gmail.com: 9/19 solved


In [29]:
analysis_id = results[0]["id"]

analysis_id

'11c680bc-bf69-4134-b216-50b701562eaa'

In [ ]:
# Get first student's analysis_id
analysis_id = results[0]["id"]
student_email = results[0]["student_email"]

# Download PDF
r = requests.get(f"{BASE}/analysis?analysis_id={analysis_id}")
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('content-type')}")
print(f"Content-Disposition: {r.headers.get('content-disposition')}")
print(r.content[:500]) 

Status: 200
Content-Type: application/json
Content-Disposition: None
b'[{"id":"11c680bc-bf69-4134-b216-50b701562eaa","student_id":"c9efa19c-c395-468d-9e3e-45b04865243c","student_email":"mathstudent2@gmail.com","homework_id":"fc23d5e4-4f6f-4f32-b95c-eee747063335","total_question":19,"total_attempted":6,"total_solved":6,"total_errors":0,"created_at":"2026-03-30T05:42:16.953921","question_evaluations":[{"question_number":1,"attempted":true,"solved":true,"error_type":null},{"question_number":2,"attempted":true,"solved":true,"error_type":null},{"question_number":3,"atte'


## 8. Export single student analysis as PDF

In [35]:
# Get first student's analysis_id
analysis_id = results[0]["id"]
student_email = results[0]["student_email"]

# Download PDF
r = requests.get(f"{BASE}/analysis/export/{analysis_id}")
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('content-type')}")
print(f"Content-Disposition: {r.headers.get('content-disposition')}")

# Save to file
if r.status_code == 200:
    filename = f"/tmp/{student_email}_homework_analysis.pdf"
    with open(filename, "wb") as f:
        f.write(r.content)
    print(f"✓ Saved to {filename}")
else:
    print(f"Error: {r.text}")

Status: 200
Content-Type: application/pdf
Content-Disposition: attachment; filename=mathstudent2@gmail.com_homework5_analysis.pdf
✓ Saved to /tmp/mathstudent2@gmail.com_homework_analysis.pdf


## 9. Export all student analyses as ZIP

In [37]:
import zipfile

# Download ZIP file with all student PDFs
r = requests.get(f"{BASE}/analysis/export-homework/", params={"homework_id": homework_id})
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('content-type')}")
print(f"Content-Disposition: {r.headers.get('content-disposition')}")
print(f"File size: {len(r.content) / 1024:.2f} KB")

# Save to file
if r.status_code == 200:
    zip_filename = f"/tmp/homework_{homework_id[:8]}_analyses.zip"
    with open(zip_filename, "wb") as f:
        f.write(r.content)
    print(f"✓ Saved to {zip_filename}")
    
    # List contents
    with zipfile.ZipFile(zip_filename, 'r') as z:
        print(f"\nZIP contents ({len(z.namelist())} files):")
        for name in z.namelist():
            info = z.getinfo(name)
            print(f"  - {name} ({info.file_size / 1024:.2f} KB)")
else:
    print(f"Error: {r.text}")

Status: 200
Content-Type: application/zip
Content-Disposition: attachment; filename=homework5_analyses.zip
File size: 7.89 KB
✓ Saved to /tmp/homework_fc23d5e4_analyses.zip

ZIP contents (2 files):
  - mathstudent2@gmail.com_homework5_analysis.pdf (6.19 KB)
  - mathstudent1@gmail.com_homework5_analysis.pdf (6.20 KB)


## 10. Generate additional practice questions

In [41]:
# Generate additional practice questions for a student based on weak topics

homework_id = results[0]["homework_id"]

r = requests.post(
    f"{BASE}/practice/generate",
    params={
        "homework_id": homework_id,
        "num_questions": 5,  # Generate 5 practice questions
        "difficulty": "similar"  # Options: easier, similar, harder
    }
)

print(f"Status: {r.status_code}")
print(f"Response: {r.json()}")
practice_job_id = r.json().get("job_id")
print(f"Practice generation job_id: {practice_job_id}")

Status: 200
Response: {'job_id': '17216b28-c035-4fb3-bc23-40b28d1a97d8', 'status': 'queued', 'message': 'Practice generation started. Poll GET /pipeline/status/{job_id} for progress.'}
Practice generation job_id: 17216b28-c035-4fb3-bc23-40b28d1a97d8


In [62]:
# Get practice question
r = requests.get(
    f"{BASE}/practice/?homework_id={homework_id}"
)
print(f"Status: {r.status_code}")
print(r.json())
practice_id = r.json()[0].get("id")
print(f"Practice question job_id: {practice_id}")

Status: 200
[{'id': '2c656018-3307-4ff0-9381-726c6f77ee59', 'user_id': None, 'homework_id': 'fc23d5e4-4f6f-4f32-b95c-eee747063335', 'group_id': '1cf1b19f-f173-4f64-a050-338a54924325', 'source': 'ai_generated', 'status': 'pending', 'version_number': 1, 'problem_data': "**1.** Evaluate the following limit:  \n$$\\lim_{x \\to \\infty} \\frac{2e^{3x} + 5e^x}{7e^{3x} - 4e^x}.$$  \n\n**2.** Evaluate the following limit:  \n$$\\lim_{x \\to 0^+} \\frac{\\ln(x)}{x^2}.$$  \n\n**3.** Find all critical numbers for the function:  \n$$f(x) = x^3 - 6x^2 + 9x.$$  \n\n**4.** Find all critical numbers for the function:  \n$$f(x) = x^2 e^x.$$  \n\n**5.** Use the First Derivative Test to determine the local extrema of the function:  \n$$f(x) = x^3 - 3x^2.$$  \n\n**6.** Evaluate the following limit using L'Hôpital's Rule:  \n$$\\lim_{x \\to 0} \\frac{\\sin(x) - x}{x^3}.$$  \n\n**7.** Evaluate the following limit:  \n$$\\lim_{x \\to \\infty} \\left(1 + \\frac{1}{x}\\right)^x.$$  \n\n**8.** Sketch the graph 

## 11. Assign additional questions

In [70]:
# Check status of assigned practice questions
r = requests.post(
    f"{BASE}/assignment/assign?practice_id={practice_id}"
)

print(f"Status: {r.status_code}")
assignment = r.json()
print(f"\nAssignment Details:")
print(assignment)

Status: 200

Assignment Details:
{'practice_id': '2c656018-3307-4ff0-9381-726c6f77ee59', 'homework_id': 'fc23d5e4-4f6f-4f32-b95c-eee747063335', 'assigned_count': 2, 'students': [{'student_id': 'c9efa19c-c395-468d-9e3e-45b04865243c', 'student_email': 'mathstudent2@gmail.com', 'weak_topics_count': 13, 'assigned_count': 10, 'fallback_used': False}, {'student_id': 'cd6cfa15-7960-47b6-a802-7daf8bea8609', 'student_email': 'mathstudent1@gmail.com', 'weak_topics_count': 11, 'assigned_count': 10, 'fallback_used': False}]}
